# Traffic Sign Visibility Audit (GTSRB)
Rule-based audit with NumPy, pandas, Matplotlib and OpenCV. **No deep learning.** Flags are triage aids, not safety verdicts. Run all cells from top to bottom.

In [ ]:
import sys, logging
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import traffic_sign_audit as tsa
from report import write_report
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

## 1. Configuration (project choices)

In [ ]:
DATA_ROOT = ROOT / 'data' / 'GTSRB'      # change to data/GTSRB_demo only for smoke tests
SELECTED = [0, 1, 13, 14, 17, 25]         # >= 5 classes
audit = tsa.TrafficSignAudit(DATA_ROOT, ROOT / 'outputs', SELECTED, max_per_class=500, seed=42)

## 2. Load and inspect the FULL annotations (full-dataset counts)

In [ ]:
ann = audit.load_annotations()
display(ann.head())
print('Rows:', audit.quality['n_rows'], '| classes:', audit.quality['n_classes'])
print('dtypes:', audit.quality['dtypes'])
pd.Series({k: v for k, v in audit.quality.items() if k.startswith('n_') or k == 'missing_values'})

In [ ]:
counts = audit.class_counts()
counts.sort_values('n_full', ascending=False).head(43)

## 3. Stratified sample, OpenCV features, tidy table

In [ ]:
audit.select_sample()
feats = audit.extract_features()
print('Skipped:', dict(audit.skipped))
pd.DataFrame(audit.exclusions).head(20)

In [ ]:
feats[['image_path','class_id','width','height','brightness_gray','contrast_gray','sharpness_lapvar','roi_valid']].head()

## 4. Review-priority rule (NumPy) and sensitivity check

In [ ]:
audit.assign_review_priority()
audit.summarize()
print(audit.thresholds)
audit.sensitivity

In [ ]:
print('Flagged overall:', int(audit.features['review_flag'].sum()), 'of', len(audit.features))
audit.flag_rate_wide

## 5. pandas summaries

In [ ]:
display(audit.class_summary)
display(audit.effect_sizes)
display(audit.rank_table)
display(audit.correlations)
audit.roi_comparison

## 6. Figures

In [ ]:
audit.plot_contact_sheet(show=True)

In [ ]:
audit.plot_distributions(show=True)

In [ ]:
audit.plot_boxplots(show=True)

In [ ]:
audit.plot_flag_rates(show=True)

In [ ]:
audit.plot_size_contrast(show=True)

In [ ]:
audit.plot_relationships(show=True)

In [ ]:
audit.plot_examples(show=True)

In [ ]:
audit.plot_roi_vs_full(show=True)

## 7. Validation cell

In [ ]:
checks = tsa.run_validation_checks(audit)
checks

## 8. Save tables and write the report

In [ ]:
audit.save_tables()
print(write_report(audit))